# Databricks notebook source
### Data Reading

In [ ]:
df1=spark.read.format("csv").option("header","true").option("inferSchema","true").load("/Volumes/workspace/default/tutorial_learning/raw/customers.csv")

In [ ]:
df1.display()

In [ ]:
df2=spark.read.format("csv").option("header","true").option("inferSchema","true").load("/Volumes/workspace/default/tutorial_learning/raw/orders.csv")
df2.display()

In [ ]:
df3=spark.read.format("csv").option("header",True).option("inferSchema",True).load("/Volumes/workspace/default/tutorial_learning/raw/order_items.csv")
df3.display()

In [ ]:
# DBTITLE 1,Read products
df4=spark.read.format("csv").option("header",True).option("inferSchema",True).load("/Volumes/workspace/default/tutorial_learning/raw/products.csv")
df4.display()

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# DBTITLE 1,Order value rollup
### Order value rollup

In [ ]:
df_ord_price=df3.groupBy('order_id').agg(round(sum(col('unit_price')*col('quantity')),0).alias("total_price"))
df_ord_price.display()

In [ ]:
df_ord_price.printSchema()

# DBTITLE 1,Validate data types
###  Validate data types

In [ ]:
# DBTITLE 1,Customers schema
df1.printSchema()

In [ ]:
df2.printSchema()

In [ ]:
df3.printSchema()

In [ ]:
df4.printSchema()

# DBTITLE 1,Check duplicates in all four tables
### Check duplicates in all four tables

In [ ]:
df1.createOrReplaceTempView("customers")
df2.createOrReplaceTempView("orders")
df3.createOrReplaceTempView("order_items")
df4.createOrReplaceTempView("products")
df_ord_price.createOrReplaceTempView("order_price")

In [ ]:
print("customers",df1.count())
print("orders",df2.count())
print("order_items",df3.count())
print("products",df4.count())

In [ ]:
print("customers",df1.distinct().count())
print("orders",df2.distinct().count())
print("order_items",df3.distinct().count())
print("products",df4.distinct().count())

# DBTITLE 1,Duplicate records note
#### 15 duplicate records in customers table

In [ ]:
%sql
select customer_id,count(customer_id) from customers group by customer_id  having count(customer_id)>1

# DBTITLE 1,Joining all tables
### Joining all tables to get complete info

In [ ]:
# DBTITLE 1,Join all tables (df)
%sql --name df
select
c.customer_id,
c.customer_name,
c.city,
c.state,
c.country,
c.signup_date,
o.order_id,
o.order_date,
o.payment_method,
o.order_status,
i.product_id,
i.quantity,
i.unit_price,
p.product_name,
p.category,
p.price
from customers as c
left join orders as o on c.customer_id = o.customer_id
left join order_items as i on o.order_id = i.order_id
left join products as p on i.product_id = p.product_id


### Phase 1: Order and sales analysis

# DBTITLE 1,Order-level dataset
#### Order-level dataset (orders + order value)

In [ ]:
# DBTITLE 1,Order-level dataset (dfo)
%sql --name dfo
select o.*,op.total_price from orders as o left join order_price as op on o.order_id = op.order_id

In [ ]:
dfo1=dfo.withColumn('month',month('order_date'))\
    .withColumn('year',year('order_date'))

In [ ]:
dfo1.display()

# DBTITLE 1,Monthly Revenue
#### Monthly Revenue

In [ ]:
df_monthly=dfo1.filter(col('order_status').isin(['Shipped','Processing','Completed'])).groupby('year','month').agg(countDistinct('order_id').alias('total_orders'),sum('total_price').alias('total_revenue')).orderBy('year','month')

In [ ]:
df_monthly.display()

#### Yearly Revenue

In [ ]:
# DBTITLE 1,Yearly Revenue (excluding Cancelled/Returned)
df_yearly=dfo1.filter(col('year')<=2025).filter(col('order_status').isin(['Shipped','Processing','Completed'])).groupby('year').agg(sum('total_price').alias('total_revenue'),countDistinct('order_id').alias('total_orders')).orderBy('year')

In [ ]:
df_yearly.withColumn('AOV',round(col('total_revenue')/col('total_orders'),2)).display()

#### Order Status Analysis

In [ ]:
df.groupby('order_status').agg(count('customer_id').alias('count')).display()

#### Payment method analysis

In [ ]:
dfo1.groupby('payment_method').agg(countDistinct('order_id').alias('total_orders'),sum('total_price').alias('total_revenue')).display()

### Phase 2: Product analysis

In [ ]:
# DBTITLE 1,Product-level dataset (dfo2)
%sql --name dfo2
select oi.*,o.order_status,p.product_name,p.category,p.price from order_items as oi left join orders as o on oi.order_id = o.order_id left join products as p on oi.product_id = p.product_id

#### Top 10 products by revenue

In [ ]:
dfo2.filter(col('order_status').isin(['Shipped','Processing','Completed'])).groupby('product_name').agg(sum('quantity').alias('total_quantity'),round(sum(col('quantity')*col('unit_price')),0).alias('total_revenue')).orderBy(col('total_revenue').desc()).limit(10).display()

#### Category performance

In [ ]:
from pyspark.sql.window import Window

In [ ]:
# DBTITLE 1,Category performance
dfo2.filter(col('order_status').isin(['Shipped','Processing','Completed'])).groupby('category').agg(sum('quantity').alias('total_quantity'),countDistinct('order_id').alias('total_orders'),round(sum(col('quantity')*col('unit_price')),0).alias('total_revenue')).withColumn('ranking',rank().over(Window.partitionBy().orderBy(col('total_revenue').desc()))).display()

### Phase 3: Customer analysis

# DBTITLE 1,Joined data overview
#### Overview of the joined data

In [ ]:
df.display()

#### Top customers by spending

In [ ]:
df.filter(col('order_status').isin(['Shipped','Processing','Completed'])).groupby('customer_id','customer_name').agg(countDistinct('order_id').alias('total_orders'),round(sum((col('quantity')*col('unit_price'))),2).alias('total_spent')).orderBy(col('total_spent').desc()).limit(10).display()

#### Sales by state or city

In [ ]:
df.filter(col('order_status').isin(['Shipped','Processing','Completed'])).groupby('state').agg(countDistinct('order_id').alias('total_orders'),sum('quantity').alias('total_quantity'),round(sum((col('quantity')*col('unit_price'))),0).alias('total_revenue')).orderBy(col('total_revenue').desc()).display()

In [ ]:
df.filter(col('order_status').isin(['Shipped','Processing','Completed'])).groupby('city').agg(countDistinct('order_id').alias('total_orders'),sum('quantity').alias('total_quantity'),round(sum((col('quantity')*col('unit_price'))),0).alias('total_revenue')).orderBy(col('total_revenue').desc()).display()

# DBTITLE 1,Phase 4: Advanced analysis
### Phase 4: Advanced analysis

#### Cancelled orders and potential lost sales

In [ ]:
df.filter(col('order_status').isin(['Cancelled','Completed'])).groupby('order_status').agg(countDistinct('order_id').alias('total_orders'),round(sum((col('quantity')*col('unit_price'))),0).alias('total_revenue')).display()

#### Customer signup cohort

In [ ]:
%sql --name df_info
select c.*,order_id,o.order_date,o.payment_method,o.order_status from customers as c left join orders as o on c.customer_id = o.customer_id

In [ ]:
# DBTITLE 1,Signup cohort: month and year columns
df_info=df_info.withColumn('signup_month',month('signup_date'))\
        .withColumn('signupyear',year('signup_date'))

In [ ]:
df_info.groupby('signupyear','signup_month').agg(countDistinct('customer_id').alias('total_customers')).orderBy('signupyear','signup_month').display()

In [ ]:
df_info.groupby('signupyear', 'signup_month').agg(
    countDistinct('customer_id').alias('total_customers'),
    countDistinct(when(col('order_id').isNotNull(), col('customer_id'))).alias('customers_who_ordered')
).withColumn(
    'conversion_rate',
    round(col('customers_who_ordered') / col('total_customers') * 100, 2)
).orderBy('signupyear', 'signup_month').display()

# DBTITLE 1,README
# 🛒 Ecommerce Sales Analytics — PySpark ETL Project

> End-to-end **ETL + analytics pipeline** on Databricks: raw e-commerce CSVs → validated & joined
> DataFrames → four phases of business analysis (sales, products, customers, cohorts).

---

## 📦 Dataset

Raw CSVs loaded from the Unity Catalog volume `workspace.default.tutorial_learning`:

| File | Grain | Rows | Key Columns |
|---|---|---:|---|
| `customers.csv` | one row per customer record | 2,015 *(2,000 distinct)* | `customer_id`, `customer_name`, `city`, `state`, `country`, `signup_date` |
| `orders.csv` | one row per order | 10,000 | `order_id`, `customer_id`, `order_date`, `payment_method`, `order_status` |
| `order_items.csv` | one row per product per order | 21,968 | `order_id`, `product_id`, `quantity`, `unit_price` |
| `products.csv` | one row per product | 100 | `product_id`, `product_name`, `category`, `price` |

**Coverage** — signups Jan 2022 → Jun 2024 · orders Jan 2022 → Jan 2026 · all customers in India · 7 product categories · 5 payment methods.

## 🔁 Pipeline

```
UC Volume (CSV) ─▶ Spark DataFrames ─▶ schema validation (printSchema)
└▶ duplicate audit  →  ⚠ 15 duplicate rows in customers
├─▶ order-level value rollup:  Σ (quantity × unit_price)  per order_id
└─▶ star join via SQL temp views:  customers ⋈ orders ⋈ order_items ⋈ products
│
▼
Phase 1: Sales  ·  Phase 2: Products  ·  Phase 3: Customers  ·  Phase 4: Advanced
```

**Data-quality rules applied**
* Revenue metrics count only `Shipped`, `Processing`, `Completed` orders — `Cancelled` / `Returned` are excluded.
* Yearly totals are capped at `year ≤ 2025` (a handful of orders fall in Jan 2026).

## 📊 Analysis Phases

| Phase | Sections | Questions answered |
|---|---|---|
| **1 — Orders & Sales** | Monthly revenue · Yearly revenue + AOV · Order status · Payment methods | How do orders and revenue trend over time? |
| **2 — Products** | Top 10 products · Category performance with `rank()` window | Which products and categories drive revenue? |
| **3 — Customers** | Top spenders · Sales by state & city | Who and where are the best customers? |
| **4 — Advanced** | Cancelled orders & lost sales · Signup cohorts + conversion rate | How much revenue leaks to cancellations? Do cohorts convert? |

## 💡 Key Insights

**Yearly performance** (valid orders, year ≤ 2025):

| Year | Orders | Revenue | AOV |
|---|---:|---:|---:|
| 2022 | 440 | $4.3M | $9.7K |
| 2023 | 1,564 | $15.9M | $10.2K |
| 2024 | 3,162 | $32.5M | $10.3K |
| 2025 | 3,363 | $39.0M | $11.6K |

* 📈 **~9× revenue growth** from 2022 to 2025, with AOV rising from ~$9.7K to ~$11.6K.
* 🖥️ **Electronics dominates** — $63.6M ≈ **67%** of product revenue; next category (Clothing) is ~$9.8M.
* 📱 **UPI is the #1 payment method** — 3,823 orders (~44% of valid orders) and $40.7M revenue.
* 🛑 **~$12.4M potential lost sales** — 690 cancelled orders ($7.5M) + 507 returned orders ($4.9M).
* 🧑‍🤝‍🧑 **99.4% conversion** — 1,987 of 2,000 customers placed at least one order.
* 🗺️ **Gujarat is the top state** ($11.1M), followed by Uttar Pradesh, Maharashtra, Punjab, and West Bengal.

## 🧰 Tech Stack

* **Databricks Serverless** compute
* **PySpark** DataFrame API — `filter`, `groupBy().agg()`, `countDistinct`, `when`, `withColumn`
* **Spark SQL** — temp views + `--name` result DataFrames for joins and cohort analysis
* **Window functions** — `rank() over Window.partitionBy() ... orderBy()`
* **Unity Catalog Volumes** for raw-file storage

## 🚀 How to Run

1. Attach a **Serverless** compute to this notebook.
2. Run all cells top-to-bottom — every step rebuilds from the raw CSVs.
3. Requires read access to `/Volumes/workspace/default/tutorial_learning/raw/`.

## ⚠ Data Quality Notes

* 15 duplicate rows in `customers` (2,015 rows vs 2,000 distinct IDs) — flagged during the duplicate audit, not dropped.
* ~$0.4M of revenue maps to a `NULL` category — order items referencing products missing from `products.csv`.
* 52 orders have a `NULL` payment method (~$0.5M revenue).
